# connections-rl — held-out eval on Kaggle GPU

Serves base + SFT + GRPO adapters through one vLLM server and runs the
paired eval on the 162-puzzle chronological test split.

Settings → Accelerator → **GPU T4 x2**, Internet → On.
Interactive session is fine (~30–45 min total). When cell 3 finishes,
**download `/kaggle/working/results.zip` immediately** — the VM is ephemeral.

In [ ]:
# Cell 1 — setup: repo, data, adapters from the Hub model registry
import os
os.environ['HF_TOKEN'] = 'hf_...'   # read token is enough; or Kaggle Add-ons > Secrets
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e . openai vllm
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local

os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

from huggingface_hub import snapshot_download
for name in ['sft', 'grpo']:
    snapshot_download(f'{HF_USER}/connections-rl-{name}',
                      local_dir=f'adapters/{name}', token=os.environ['HF_TOKEN'])
!ls adapters/sft adapters/grpo

In [ ]:
# Cell 2 — vLLM OpenAI-compatible server with both LoRA adapters mounted
import subprocess, time, urllib.request

A = '/kaggle/working/connections-rl/adapters'
proc = subprocess.Popen(
    f'vllm serve Qwen/Qwen2.5-1.5B-Instruct --enable-lora '
    f'--lora-modules connections-rl-sft={A}/sft connections-rl-grpo={A}/grpo '
    f'--max-model-len 2048 --gpu-memory-utilization 0.85',
    shell=True,
    stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(120):  # model download + startup can take ~5-10 min
    try:
        urllib.request.urlopen('http://localhost:8000/health')
        print('vLLM ready')
        break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError('vLLM failed — check /kaggle/working/vllm.log')

In [ ]:
# Cell 3 — run the full eval (base / sft / grpo, paired stats) and bundle results
!python -m connections_rl.eval.run --config configs/eval/default.yaml
!zip -qr /kaggle/working/results.zip results
print('done — download /kaggle/working/results.zip from the output panel')

In [ ]:
# Cell 4 (optional) — peek at a few GRPO completions on test puzzles
import json, random
from openai import OpenAI
from connections_rl.data.formatting import build_chat
from connections_rl.data.loader import load_puzzles

client = OpenAI(base_url='http://localhost:8000/v1', api_key='none')
puzzles = load_puzzles('data/splits/puzzles_test.json')
for p in random.Random(0).sample(puzzles, 3):
    out = client.chat.completions.create(
        model='connections-rl-grpo', messages=build_chat(p),
        temperature=0.0, max_tokens=256)
    print(f'--- puzzle {p.puzzle_id} ({p.date}) ---')
    print(out.choices[0].message.content)
    print()